# ATU: Automated Trading Unit - Master Google Colab GPU Workflow

This notebook integrates the **ATU** Reinforcement Learning trading system with Google Colab GPU compute, GitHub source synchronization, and Google Drive persistent storage.

### Pipeline Architecture:
- **Development Workstation:** Google Antigravity IDE (Windows)
- **Version Control & Synchronization:** GitHub (`https://github.com/SUZUB/ATU.git`)
- **Compute Runtime:** Google Colab GPU (NVIDIA T4 / V100 / A100)
- **Persistent Storage:** Google Drive (`/content/drive/MyDrive/ATU_Models/` & `/content/drive/MyDrive/ATU_Data/`)

### 20-Section Execution Workflow:
1. Environment Initialization
2. Python Version Verification
3. GPU Detection
4. NVIDIA GPU Information
5. PyTorch CUDA Verification
6. GitHub Repository Synchronization
7. Dependency Installation
8. Google Drive Mounting
9. Dataset Validation (/content/drive/MyDrive/ATU_Data/)
10. Project Initialization & Module Imports
11. FinBERT Sentiment Initialization
12. Market Data Processing (5-min OHLCV)
13. Feature Engineering (Technical Indicators + Sentiment)
14. PPO Environment Initialization (Gymnasium Rules Engine)
15. Training Phase: 1,000-Step Initial Smoke Test
16. Full Training Execution (200,000 Timesteps with Checkpoints)
17. Evaluation & Out-of-Sample Continuous Backtest
18. Institutional Financial Metrics Generation
19. Model Checkpoint Saving & Verification
20. Logs, Results Persistence & Download

## 1. Environment Initialization
Prepare the working directories and runtime environment.

In [ ]:
import os
import sys
from pathlib import Path

WORK_DIR = Path("/content/ATU")
print(f"Workspace Root: {WORK_DIR}")
print(f"System Path:    {sys.path[:3]}")

## 2. Python Version Verification
Confirm Python version compatibility (>= 3.10 required).

In [ ]:
import platform

py_version = sys.version.split()[0]
print(f"Python Version: {py_version}")
print(f"Platform:       {platform.platform()}")
assert sys.version_info >= (3, 10), f"Python 3.10+ required, found {py_version}"
print("Python runtime check passed.")

## 3. GPU Detection
Check for active CUDA GPU runtime in Google Colab.

In [ ]:
import torch

cuda_is_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_is_available}")
if not cuda_is_available:
    print("WARNING: No GPU detected. In Colab menu: Runtime -> Change runtime type -> T4 GPU")
else:
    print("GPU detected successfully!")

## 4. NVIDIA GPU Information
Query GPU hardware specifications, VRAM, and driver details.

In [ ]:
if torch.cuda.is_available():
    dev_idx = torch.cuda.current_device()
    dev_name = torch.cuda.get_device_name(dev_idx)
    vram_bytes = torch.cuda.get_device_properties(dev_idx).total_memory
    vram_gb = vram_bytes / (1024 ** 3)
    print(f"Active GPU:   {dev_name}")
    print(f"Total VRAM:   {vram_gb:.2f} GB")
    !nvidia-smi
else:
    print("Running in CPU mode. Skipping nvidia-smi.")

## 5. PyTorch CUDA Verification
Verify tensor allocation and CUDA kernel execution.

In [ ]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Version:    {torch.version.cuda}")

if torch.cuda.is_available():
    t = torch.randn((1000, 1000), device="cuda")
    t_prod = torch.matmul(t, t)
    print(f"PyTorch CUDA tensor benchmark passed. Device: {t_prod.device}")
else:
    t = torch.randn((100, 100))
    print(f"PyTorch CPU tensor test passed.")

## 6. GitHub Repository Synchronization
Clone or pull the latest code from `https://github.com/SUZUB/ATU.git`.

In [ ]:
# SECTION 6: Secure GitHub Synchronization
# Uses a fine-grained, read-only GitHub token from Colab Secrets.

import os
import subprocess
import base64
from pathlib import Path
from google.colab import userdata

REPO_URL = "https://github.com/SUZUB/ATU.git"
PROJECT_DIR = Path("/content/ATU")

# Retrieve token securely. Never print it.
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    raise ValueError("GITHUB_TOKEN is missing from Colab Secrets.")

# Configure temporary authentication for Git subprocesses.
auth_string = base64.b64encode(
    f"x-access-token:{GITHUB_TOKEN}".encode()
).decode()

git_env = os.environ.copy()
git_env["GIT_CONFIG_COUNT"] = "1"
git_env["GIT_CONFIG_KEY_0"] = "http.extraheader"
git_env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth_string}"

# Clone or update the repository.
if not PROJECT_DIR.exists():
    print("Cloning private ATU repository...")

    result = subprocess.run(
        ["git", "clone", "--branch", "main", REPO_URL, str(PROJECT_DIR)],
        env=git_env,
        capture_output=True,
        text=True
    )
else:
    print("Updating existing ATU repository...")

    result = subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "pull", "origin", "main"],
        env=git_env,
        capture_output=True,
        text=True
    )

if result.returncode != 0:
    raise RuntimeError(
        "Git synchronization failed:\n" + result.stderr
    )

# Remove the secret from the Python variable after use.
del GITHUB_TOKEN
del auth_string
del git_env

print("ATU GitHub synchronization successful!")

# Verify the latest commit.
subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "log", "-1", "--oneline"],
    check=True
)

%cd /content/ATU

## 7. Dependency Installation
Install pinned production dependencies from `requirements.txt`.

In [ ]:
%cd {PROJECT_DIR}
!pip install -q -r requirements.txt
print("Dependencies installed successfully.")

## 8. Google Drive Mounting
Mount Google Drive for persistent models (`/content/drive/MyDrive/ATU_Models/`) and datasets (`/content/drive/MyDrive/ATU_Data/`).

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

DRIVE_MODELS_DIR = Path('/content/drive/MyDrive/ATU_Models')
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/ATU_Data')

# Organize output directory tree
for sub in ["checkpoints", "final_models", "training_logs", "backtest_results", "evaluation_metrics"]:
    (DRIVE_MODELS_DIR / sub).mkdir(parents=True, exist_ok=True)
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Persistent Models Directory:  {DRIVE_MODELS_DIR}")
print(f"Persistent Datasets Directory: {DRIVE_DATA_DIR}")
!ls -ld /content/drive/MyDrive/ATU_Models /content/drive/MyDrive/ATU_Data

## 9. Dataset Validation (/content/drive/MyDrive/ATU_Data/)
Inspect and validate the historical financial news datasets in Google Drive.

In [ ]:
import pandas as pd
from pathlib import Path

dataset_path = DRIVE_DATA_DIR / "analyst_ratings_processed.csv"
print(f"Target Dataset Path: {dataset_path}")

# 1. Verify existence & non-zero size
if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset file missing: {dataset_path}\n"
        f"Please upload 'analyst_ratings_processed.csv' to your Google Drive under 'My Drive/ATU_Data/'."
    )

file_size_bytes = dataset_path.stat().st_size
file_size_mb = file_size_bytes / (1024 * 1024)
print(f"File Size: {file_size_mb:.2f} MB ({file_size_bytes:,} bytes)")
if file_size_bytes == 0:
    raise ValueError(f"Dataset file exists but has 0 bytes: {dataset_path}")

# 2. Inspect sample without loading entire 150MB file
sample_df = pd.read_csv(dataset_path, nrows=20)
print(f"Sample Rows Read: {len(sample_df)}")
print(f"Dataset Columns:  {list(sample_df.columns)}")

# 3. Verify columns match data processor expectations ('title' or 'headline', 'date', 'stock')
headline_col = "title" if "title" in sample_df.columns else ("headline" if "headline" in sample_df.columns else None)
date_col = "date" if "date" in sample_df.columns else None
stock_col = next((c for c in sample_df.columns if c.lower() in ["stock", "ticker"]), None)

if not headline_col:
    raise KeyError(f"Expected headline column ('title' or 'headline') not found in {list(sample_df.columns)}")
if not date_col:
    raise KeyError(f"Expected date column ('date') not found in {list(sample_df.columns)}")
if not stock_col:
    raise KeyError(f"Expected stock column ('stock' or 'ticker') not found in {list(sample_df.columns)}")

print(f"Column Mapping: headline='{headline_col}', date='{date_col}', stock='{stock_col}'")

# 4. Verify date interpretation & timezone handling
parsed_dates = pd.to_datetime(sample_df[date_col].dropna(), utc=True, errors="coerce")
if parsed_dates.isnull().all():
    raise ValueError(f"Failed to parse dates in sample column '{date_col}'")
print(f"Date Parsing: Verified (Sample range: {parsed_dates.min().strftime('%Y-%m-%d')} to {parsed_dates.max().strftime('%Y-%m-%d')})")

# 5. Check target ticker (AAPL) using chunked streaming
print("Scanning for 'AAPL' records using memory-efficient chunked reader...")
aapl_matches = []
for chunk in pd.read_csv(dataset_path, chunksize=25000):
    matches = chunk[chunk[stock_col].astype(str).str.upper() == "AAPL"]
    if not matches.empty:
        aapl_matches.append(matches)
    if len(aapl_matches) >= 5 or sum(len(m) for m in aapl_matches) >= 500:
        break

if aapl_matches:
    aapl_df = pd.concat(aapl_matches, ignore_index=True)
    aapl_dates = pd.to_datetime(aapl_df[date_col], utc=True, errors="coerce").dt.strftime("%Y-%m-%d")
    print(f"AAPL Records Found: {len(aapl_df):,} headlines in scanned chunks")
    print(f"AAPL Date Span:     {aapl_dates.min()} to {aapl_dates.max()}")
    print("Sample AAPL Headline:")
    print(f"  \"{aapl_df[headline_col].iloc[0]}\"")
else:
    raise ValueError("No records found for ticker 'AAPL' in scanned dataset chunks.")

print("\nDataset validation PASSED: Real financial news dataset is verified and ready for FinBERT!")

## 10. Project Initialization & Module Imports
Import and verify all 4 ATU core components.

In [ ]:
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from market_data_processor import MarketDataProcessor
from sentiment_processor import SentimentProcessor
from constrained_trading_env import ConstrainedTradingEnv
from src.data_processing import (
    create_final_state_vector,
    generate_intraday_5min_market_data,
    load_archive_headlines,
)
from src.model import create_vectorized_env, create_ppo_agent
from train_and_backtest import run_pipeline, calculate_financial_metrics

print("All ATU components imported successfully!")

## 11. FinBERT Sentiment Initialization
Load `ProsusAI/finbert` on GPU and score sample financial headlines.

In [ ]:
import time

target_device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Initializing SentimentProcessor on: {target_device.upper()}...")

t0 = time.time()
sp = SentimentProcessor(device=target_device)
load_time = time.time() - t0
print(f"FinBERT loaded in {load_time:.2f}s on {sp.device}")

sample_headline = "Apple reports record quarterly revenue driven by strong iPhone services growth"
score = sp.get_sentiment_score(sample_headline)
print(f"Sample Headline: '{sample_headline}'")
print(f"Sentiment Score: {score:+.4f} (Directional continuous metric in [-1.0, 1.0])")

## 12. Market Data Processing (5-min OHLCV)
Process or download 5-minute intraday market bars.

In [ ]:
# Load real headlines for AAPL from Google Drive (or local archive)
headlines = load_archive_headlines(archive_dir=DRIVE_DATA_DIR, ticker="AAPL", max_rows=500, allow_synthetic_fallback=False)
print(f"Ingested {len(headlines):,} real headlines for AAPL.")

# Score sentiment with FinBERT
sentiment_df = sp.process_news_dataframe(headlines, batch_size=32)
print(f"FinBERT scored {len(headlines):,} headlines into {len(sentiment_df):,} distinct trading days.")

# Synchronize market data start date with headline date anchor
market_start_date = sentiment_df["date"].min()
print(f"Synchronizing market start date to sentiment anchor: {market_start_date}")

mdp = MarketDataProcessor()
raw_candles = generate_intraday_5min_market_data(n_rows=1500, start_date=market_start_date, initial_price=150.0)
processed_market_df = mdp.process_dataframe(raw_candles)
print(f"Market DataFrame Shape: {processed_market_df.shape}")
display(processed_market_df.head(3))

## 13. Feature Engineering (Technical Indicators + Sentiment)
Merge technical indicators (RSI, MACD, ATR, BB, ADX) with daily sentiment into `final_state_vector`.

In [ ]:
# Merge market features and FinBERT daily sentiment into unified state matrix
final_state_vector = create_final_state_vector(processed_market_df, sentiment_df)
sentiment_series = final_state_vector["daily_sentiment"]
non_zero_pct = float((sentiment_series != 0.0).mean()) * 100.0

print(f"final_state_vector Shape: {final_state_vector.shape}")
print(f"Columns: {list(final_state_vector.columns)}")
print(f"Sentiment Feature Statistics:")
print(f"  - Mean:     {sentiment_series.mean():+.4f}")
print(f"  - Std Dev:  {sentiment_series.std():.4f}")
print(f"  - Range:    [{sentiment_series.min():+.4f}, {sentiment_series.max():+.4f}]")
print(f"  - Non-Zero: {non_zero_pct:.1f}% of state vectors")

assert non_zero_pct > 0.0, "CRITICAL: Sentiment features must not be all zero!"
print("Feature engineering PASSED: Real FinBERT sentiment correctly integrated into state vectors.")

## 14. PPO Environment Initialization (Gymnasium Rules Engine)
Vectorize `ConstrainedTradingEnv` and verify observation and action spaces.

In [ ]:
# 80% Train, 20% Out-of-sample Test Split
split_idx = int(len(final_state_vector) * 0.8)
train_df = final_state_vector.iloc[:split_idx].copy()
test_df = final_state_vector.iloc[split_idx:].copy()

env = create_vectorized_env(df=train_df)
print(f"Action Space:      {env.action_space}")
print(f"Observation Space: {env.observation_space}")
print(f"Training Candles:  {len(train_df):,} | Testing Candles: {len(test_df):,}")

## 15. Training Phase: 1,000-Step Initial Smoke Test
Run an initial fast smoke test (1,000 steps) to verify reward calculation, policy stability, and Google Drive checkpoint creation.

In [ ]:
compute_device = "cuda" if torch.cuda.is_available() else "cpu"
smoke_save_path = str(DRIVE_MODELS_DIR)

print(f"Launching Initial Smoke Test (1,000 timesteps) on device: {compute_device}...")

# Run smoke test using real dataset and FinBERT (allow_synthetic_fallback omitted by default)
!python train_and_backtest.py \
    --timesteps 1000 \
    --ticker AAPL \
    --device {compute_device} \
    --model-save-path "{smoke_save_path}" \
    --data-dir "{DRIVE_DATA_DIR}" \
    --finbert

## 16. Full Training Execution (200,000 Timesteps with Checkpoints)
Execute complete 200,000-step training with periodic checkpoints saved every 25,000 steps directly to Google Drive.
> **Note:** Run this cell after reviewing the smoke test results above.

In [ ]:
# Uncomment below when ready to run the full 200,000-step training session:
# !python train_and_backtest.py \
#     --timesteps 200000 \
#     --ticker AAPL \
#     --device {compute_device} \
#     --model-save-path "{DRIVE_MODELS_DIR}" \
#     --data-dir "{DRIVE_DATA_DIR}" \
#     --checkpoint-freq 25000

## 17. Evaluation & Out-of-Sample Continuous Backtest
Evaluate the trained model across the out-of-sample test set in an unbroken continuous compounding episode.

In [ ]:
from stable_baselines3 import PPO

model_candidates = [
    DRIVE_MODELS_DIR / "final_models" / "ppo_trading_agent_v1.zip",
    DRIVE_MODELS_DIR / "ppo_trading_agent_v1.zip",
    Path("ppo_trading_agent_v1.zip"),
]
model_path = next((p for p in model_candidates if p.exists()), None)
if model_path is None:
    raise FileNotFoundError(
        "Trained model checkpoint not found in Google Drive.\n"
        "Please execute Section 15 (Initial Smoke Test) or Section 16 (Full Training) before running this backtest cell."
    )

print(f"Loading verified model checkpoint: {model_path}")
loaded_model = PPO.load(str(model_path.with_suffix("")), device=compute_device)

test_env = ConstrainedTradingEnv(df=test_df)
state, info = test_env.reset(seed=42)

portfolio_values = [test_env.initial_balance]
step_returns = []
done = False
profitable_trades = 0

while not done:
    action, _ = loaded_model.predict(state, deterministic=True)
    state, reward, terminated, truncated, step_info = test_env.step(action)
    done = terminated or truncated
    portfolio_values.append(float(step_info["balance"]))
    step_returns.append(float(step_info["step_profit"]))
    if step_info.get("trade_closed_profit", False):
        profitable_trades += 1

print(f"Backtest evaluation completed across {len(portfolio_values)} steps.")

## 18. Institutional Financial Metrics Generation
Compute Sharpe Ratio, Sortino Ratio, Maximum Drawdown, Win Rate, and Gross vs Net Returns.

In [ ]:
metrics = calculate_financial_metrics(
    portfolio_values=portfolio_values,
    step_returns=step_returns,
    total_trades=step_info.get("total_trades", 0),
    profitable_trades=profitable_trades,
    initial_balance=test_env.initial_balance,
    final_balance=float(step_info["account_balance"]),
    total_fees_paid=float(step_info["total_fees_paid"]),
)

print("=" * 65)
print("           INSTITUTIONAL PERFORMANCE METRICS REPORT           ")
print("=" * 65)
print(f" Initial Balance:        ₹{metrics['initial_balance']:,.2f}")
print(f" Final Compounded Balance:₹{metrics['final_balance']:,.2f}")
print(f" Net Profit (After Fees):₹{metrics['net_profit']:+,.2f} ({metrics['net_return_pct']:+.2f}%)")
print(f" Gross Profit (Pre-Fees):₹{metrics['gross_profit']:+,.2f} ({metrics['gross_return_pct']:+.2f}%)")
print(f" Total Brokerage & STT:  ₹{metrics['total_fees_paid']:,.2f}")
print(f" Total Trades Executed:  {metrics['total_trades']} (Win Rate: {metrics['win_rate_pct']:.1f}%)")
print(f" Maximum Drawdown:       {metrics['max_drawdown_pct']:.2f}%")
print(f" Sharpe Ratio (Ann.):    {metrics['sharpe_ratio']:.4f}")
print(f" Sortino Ratio (Ann.):   {metrics['sortino_ratio']:.4f}")
print("=" * 65)

## 19. Model Checkpoint Saving & Verification
Verify all saved files in Google Drive: checkpoints, final models, and logs.

In [ ]:
print("Google Drive Models Directory Contents:")
!ls -lR /content/drive/MyDrive/ATU_Models/

## 20. Logs, Results Persistence & Download
Visualize equity curve and provide download helpers.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(portfolio_values, color="#2563EB", linewidth=2, label="ATU Compounded Balance (₹)")
plt.axhline(test_env.initial_balance, color="gray", linestyle="--", alpha=0.7, label="Initial Capital")
plt.title("ATU Intraday Out-of-Sample Equity Curve (AAPL)", fontsize=14)
plt.xlabel("Intraday 5-Min Steps", fontsize=12)
plt.ylabel("Portfolio Balance (₹)", fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DRIVE_MODELS_DIR / "backtest_results" / "equity_curve.png", dpi=150)
plt.show()

from google.colab import files
def download_results():
    archive_file = "/content/ATU_results.zip"
    !zip -q -r {archive_file} /content/drive/MyDrive/ATU_Models/backtest_results /content/drive/MyDrive/ATU_Models/evaluation_metrics
    files.download(archive_file)

# Run download_results() to download all metrics and charts to your local computer:
# download_results()